In [33]:
import numpy as np
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [20]:
# Load the model from the .h5 file
loaded_model = joblib.load('./model/career_prediction_model.h5')
loaded_encoder = joblib.load('./model/career_label_encoder.h5')

print(f"Model type: {type(loaded_model)}")
print(f"Number of classes: {loaded_model.n_classes_}")
print("Model successfully imported and ready for prediction!")

Model type: <class 'xgboost.sklearn.XGBClassifier'>
Number of classes: 72
Model successfully imported and ready for prediction!


In [25]:
#Column bounds are set using min max values from the original dataset

column_bounds = {
    'language_skills':     {'min': 0, 'max': 20},
    'musical_ability':     {'min': 0, 'max': 20},
    'physical_prowess':    {'min': 0, 'max': 20},
    'math_and_logic':      {'min': 0, 'max': 20},
    'spatial_awareness':   {'min': 0, 'max': 20},
    'collaboration_skills':{'min': 0, 'max': 20},
    'self_awareness':      {'min': 0, 'max': 20},
    'sustainability_focus':{'min': 0, 'max': 20}
}

In [26]:
#Simulated User Inputs, using a 1-5 scale determining each of the users self evaluations of the categories

user_inputs = {
    'language_skills': 4,
    'musical_ability': 2,
    'physical_prowess': 5,
    'math_and_logic': 3,
    'spatial_awareness': 1,
    'collaboration_skills': 4,
    'self_awareness': 5,
    'sustainability_focus': 2
}

In [27]:
model_ready_vector = []

for column, user_value in user_inputs.items():
    c_min = column_bounds[column]['min']
    c_max = column_bounds[column]['max']
    
    x_original = c_min + ((user_value - 1) * (c_max - c_min) / 4)
    
    x_normalized = (x_original - c_min) / (c_max - c_min)
    
    model_ready_vector.append(x_normalized)

# Convert to numpy array for model prediction
final_input = np.array(model_ready_vector).reshape(1, -1)
print(final_input)

[[0.75 0.25 1.   0.5  0.   0.75 1.   0.25]]


In [28]:
all_probabilities = loaded_model.predict_proba(final_input)[0]

top_5_indices = np.argsort(all_probabilities)[-5:][::-1]

top_5_careers = loaded_encoder.inverse_transform(top_5_indices)

top_5_scores = all_probabilities[top_5_indices]

print("--- Top 5 Career Recommendations for a simulated input ---")
for rank, (career, score) in enumerate(zip(top_5_careers, top_5_scores), 1):
    confidence_percent = score * 100
    print(f"{rank}. {career} ({confidence_percent:.2f}%)")

--- Top 5 Career Recommendations for a simulated input ---
1. Computer analyst (71.57%)
2. Para Medical (physiotherapy, occupational theropy, audio and speech language theropy, nursing) (8.61%)
3. Company secretary (2.29%)
4. Leader (1.45%)
5. Politician (0.96%)


In [39]:
import numpy as np
import pandas as pd

def get_career_recommendations():
    print("--- Career Discovery Input (Scale 1 to 5) ---")
    print("1: Very Low | 3: Average | 5: Very High\n")
    
    user_features_normalized = []
    
    for col, bounds in column_bounds.items():
        feature_name = col.replace('_', ' ').title()
        while True:
            try:
                val = float(input(f"Rate your '{feature_name}' (1-5): "))
                if 1 <= val <= 5:
                    c_min = bounds['min']
                    c_max = bounds['max']
                    
                    x_original = c_min + ((val - 1) * (c_max - c_min) / 4)
                    
                    x_normalized = (x_original - c_min) / (c_max - c_min)
                    
                    user_features_normalized.append(x_normalized)
                    break
                else:
                    print("Please enter a value between 1 and 5.")
            except ValueError:
                print("Invalid input. Please enter a number.")

    input_matrix = np.array([user_features_normalized])
    
    probs = loaded_model.predict_proba(input_matrix)[0]
    
    top_5_indices = np.argsort(probs)[-5:][::-1]
    
    recommendations = loaded_encoder.inverse_transform(top_5_indices)

    print("\n--- Top 5 Career Matches ---")
    for i, (career, idx) in enumerate(zip(recommendations, top_5_indices), 1):
        confidence = probs[idx] * 100
        print(f"{i}. {career} ({confidence:.1f}% Match Score)")

In [40]:
#Re-run this cell to get personalized recommendations based on personal input

get_career_recommendations()

--- Career Discovery Input (Scale 1 to 5) ---
1: Very Low | 3: Average | 5: Very High



Rate your 'Language Skills' (1-5):  3
Rate your 'Musical Ability' (1-5):  2
Rate your 'Physical Prowess' (1-5):  4
Rate your 'Math And Logic' (1-5):  5
Rate your 'Spatial Awareness' (1-5):  5
Rate your 'Collaboration Skills' (1-5):  5
Rate your 'Self Awareness' (1-5):  5
Rate your 'Sustainability Focus' (1-5):  2



--- Top 5 Career Matches ---
1. Manager (81.7% Match Score)
2. Politician (8.1% Match Score)
3. Engineer (2.9% Match Score)
4. Internal auditor (1.2% Match Score)
5. Chief financial officer (1.2% Match Score)
